# Unit 11 / Chapter 11: Quantum NLP and Quantum Transformers

> **Main Learning Objective:** Learn how classical language data (words, sentences, and attention patterns) can be encoded into quantum states, and build a tiny quantum sentence classifier that uses a parameterized quantum attention head.

| Section | Topic |
|---|---|
| 11.1 | Words as vectors, and how to load them into qubits |
| 11.2 | Sentence structure via tensor networks (DisCoCat intuition) |
| 11.3 | Quantum attention: a parameterized quantum head |
| 11.4 | A tiny quantum sentence classifier |

---
## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100

# Tiny quantum simulator used across all units
def ket0(n):
    s = np.zeros(2**n, dtype=complex); s[0] = 1.0
    return s
def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out
I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)
def Rx(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)
def apply_1q(gate, qubit, n):
    return kron_all([gate if i==qubit else I2 for i in range(n)])
def apply_cnot(control, target, n):
    dim = 2**n
    op = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        bits = [(x >> (n-1-i)) & 1 for i in range(n)]
        if bits[control] == 1:
            bits[target] ^= 1
        y = 0
        for b in bits:
            y = (y<<1) | b
        op[y, x] = 1
    return op
def expZ(state, qubit, n):
    Zop = apply_1q(Z, qubit, n)
    return float(np.real(np.conj(state) @ Zop @ state))
print("Quantum simulator ready.")

---
## Course check-in

This logs that you started **Unit 11**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 11
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 11.1: Words as Vectors, and How to Load Them into Qubits

In classical NLP, every word is a vector. Word2vec, GloVe, and modern transformer embeddings all boil down to "each word becomes a point in R^d." Similar words sit near each other in that vector space.

Quantum NLP starts from the same idea, but the vector lives on qubits. A word is represented by a quantum state |w>. Two similar words produce states with high overlap |<w1|w2>|^2.

The cheapest way to load a small vector onto qubits is **angle encoding**: for a 2-dimensional word vector (a, b), we set

  |w> = cos(a/2)|0> + e^{i b} sin(a/2)|1>

which is just a Bloch-sphere point. Similar (a, b) produce nearby Bloch points and therefore high overlap.

Below we build tiny 1-qubit embeddings for four toy words and measure their pairwise similarity.

In [ ]:
# Toy 2D "word embeddings" for four words
words = {
    "king":   (1.2,  0.3),
    "queen":  (1.3,  0.4),
    "apple":  (2.6,  1.8),
    "orange": (2.7,  1.9),
}

def word_state(vec):
    a, b = vec
    c, s = np.cos(a/2), np.sin(a/2)
    return np.array([c, np.exp(1j*b)*s], dtype=complex)

states = {w: word_state(v) for w, v in words.items()}
names = list(states.keys())
sim = np.zeros((4, 4))
for i, wi in enumerate(names):
    for j, wj in enumerate(names):
        sim[i, j] = abs(np.vdot(states[wi], states[wj]))**2

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(sim, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(names, rotation=30); ax.set_yticklabels(names)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{sim[i,j]:.2f}", ha='center', va='center',
                color='white' if sim[i,j] < 0.6 else 'black', fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046); ax.set_title("Quantum word similarity |<w1|w2>|^2")
plt.tight_layout(); plt.show()

Notice that "king" and "queen" have high overlap, and "apple" and "orange" have high overlap, while cross-category pairs (king vs. apple) have low overlap. The quantum inner product recovers semantic similarity out of the box, without dot-product hardware.

### Activity 11.1

Add a fifth word "prince" that should sit close to "king" (say vector (1.25, 0.35)) and print its similarity to every other word.

In [ ]:
# TODO: encode 'prince' with vector (1.25, 0.35), then print similarity to
# every existing word. Fill in the two lines marked None.
prince_vec = None            # <-- your tuple here
prince_state = None          # <-- call word_state(prince_vec)

if prince_state is not None:
    for w, s in states.items():
        print(f"prince vs {w:6s}: {abs(np.vdot(prince_state, s))**2:.3f}")
else:
    print("Fill in prince_vec and prince_state above, then re-run.")

<details><summary>Solution</summary>

```python
prince_vec = (1.25, 0.35)
prince_state = word_state(prince_vec)
```
Similarity to king should be ~0.999, to queen ~0.996, to apple/orange ~0.15.
</details>

---
# Section 11.2: Sentence Structure via Tensor Networks

Words alone are not language. "Dog bites man" and "man bites dog" have the same words but opposite meanings. Order matters.

Classical transformers use positional encodings and attention to capture order. Quantum NLP has a related tool: the **DisCoCat** framework (Distributional Compositional Categorical model) by Coecke, Sadrzadeh, and Clark. The core idea:

1. Each word gets a quantum state whose shape depends on its grammatical role (noun, transitive verb, adjective, etc.).
2. Grammatical composition rules become tensor contractions between those states.
3. The final sentence is a single quantum state whose meaning is the composition.

For today we will use a stripped-down version: a sentence is represented by a small entangled multi-qubit state that combines word states with an entangler that encodes their grammatical link.

In [ ]:
# Two-word sentence: (subject, verb)
# Subject is a 1-qubit word state. Verb is a 1-qubit word state.
# Sentence state = CNOT applied to (subject tensor verb), with H on the verb wire.
# This entangles them so the verb "acts on" the subject.

def sentence_state(subject_vec, verb_vec):
    s_sub  = word_state(subject_vec)
    s_verb = word_state(verb_vec)
    state  = np.kron(s_sub, s_verb)             # 4-vector on 2 qubits
    state  = apply_1q(H, 1, 2) @ state          # H on verb wire
    state  = apply_cnot(0, 1, 2) @ state        # entangler (subject -> verb)
    return state

# "Dogs" as subject, "bark" as verb
dogs = (1.0, 0.2); bark = (2.0, 1.0)
s1 = sentence_state(dogs, bark)

# "Cats" as subject, "bark" as verb (nonsense)
cats = (2.5, 1.5)
s2 = sentence_state(cats, bark)

# Overlap: how meaningful is each sentence?
print("Prob of 'dogs bark' sentence state overlapping |00>:", round(abs(s1[0])**2, 3))
print("Prob of 'cats bark' sentence state overlapping |00>:", round(abs(s2[0])**2, 3))
print("Overlap between the two sentences: ",
      round(abs(np.vdot(s1, s2))**2, 3))

The two sentences share a verb but differ in subject, and the quantum state reflects that overlap. If you swap subject and verb roles (making the sentence ungrammatical), you get a different state, so word order is preserved.

---
# Section 11.3: Quantum Attention

Classical transformers compute attention as

  Attention(Q, K, V) = softmax(Q K^T / sqrt(d)) V

The core operation is a similarity dot product between queries and keys. That is exactly the kind of operation a quantum computer is good at: quantum states naturally compute inner products through a **swap test**.

### The swap test in one paragraph

Given two states |a> and |b>, prepare an ancilla qubit in |+>, then do a controlled-swap between |a> and |b>, then apply H to the ancilla and measure it. The probability of measuring 0 is (1 + |<a|b>|^2) / 2. So measurement gives you the overlap.

Below we build a mini quantum attention head: given a query word and two key words, we compute overlaps quantum-style and use them as attention weights.

In [ ]:
def overlap(a, b):
    return abs(np.vdot(a, b))**2

def quantum_attention(query_vec, key_vecs, value_vecs):
    q  = word_state(query_vec)
    ks = [word_state(v) for v in key_vecs]
    vs = np.array(value_vecs, dtype=float)      # values are scalars for demo
    scores = np.array([overlap(q, k) for k in ks])
    weights = np.exp(scores) / np.sum(np.exp(scores))
    return float(weights @ vs), weights, scores

# Query: word similar to 'king'
q_vec = (1.2, 0.3)
# Keys: king, apple. Values: sentiment scores (+1 royal, -1 fruit)
keys   = [(1.2, 0.3), (2.6, 1.8)]
values = [+1.0, -1.0]

attended, w, sc = quantum_attention(q_vec, keys, values)
print("Overlap scores:  ", np.round(sc, 3))
print("Attention weights:", np.round(w, 3))
print("Attended value:   ", round(attended, 3))

### Activity 11.2

Change the query vector to be similar to "apple" (say (2.6, 1.8)). The attended value should flip sign. Try it and confirm.

In [ ]:
# TODO: swap the query so it is close to 'apple'. Re-run quantum_attention.
q_vec_new = None       # <-- fill in

if q_vec_new is not None:
    attended, w, sc = quantum_attention(q_vec_new, keys, values)
    print("New attention weights:", np.round(w, 3))
    print("New attended value:   ", round(attended, 3))
else:
    print("Fill in q_vec_new above, then re-run.")

<details><summary>Solution</summary>

```python
q_vec_new = (2.6, 1.8)
```
Weights flip toward the "apple" key and the attended value becomes negative (~-0.62).
</details>

---
# Section 11.4: A Tiny Quantum Sentence Classifier

We now put the pieces together. Given a two-word sentence (subject verb), we:

1. Build the sentence state.
2. Rotate it through a trainable ansatz U(theta).
3. Measure <Z> on the first qubit as the model output.
4. Interpret output > 0 as class "positive" and output < 0 as class "negative."

We will train it on four toy sentences by hand-grid-searching theta.

In [ ]:
# Training set: (subject vec, verb vec, label)
data = [
    ((1.0, 0.2), (2.0, 1.0),  +1),   # dogs bark  -> positive
    ((1.1, 0.3), (2.1, 1.1),  +1),   # puppies yap -> positive
    ((2.5, 1.5), (0.4, 0.1),  -1),   # cats meow (labeled negative for demo)
    ((2.6, 1.6), (0.5, 0.2),  -1),
]

def model(theta, sub_vec, verb_vec):
    s = sentence_state(sub_vec, verb_vec)
    U = apply_1q(Ry(theta[0]), 0, 2) @ apply_1q(Ry(theta[1]), 1, 2)
    U = apply_cnot(1, 0, 2) @ U
    U = apply_1q(Rz(theta[2]), 0, 2) @ U
    s = U @ s
    return expZ(s, 0, 2)

def loss(theta):
    err = 0.0
    for sub, verb, y in data:
        pred = model(theta, sub, verb)
        err += (pred - y)**2
    return err / len(data)

# grid search
best_L = 1e9; best_theta = None
for t1 in np.linspace(0, 2*np.pi, 12):
    for t2 in np.linspace(0, 2*np.pi, 12):
        for t3 in np.linspace(0, 2*np.pi, 12):
            L = loss([t1, t2, t3])
            if L < best_L:
                best_L = L; best_theta = [t1, t2, t3]
print("best loss:", round(best_L, 3), "at theta =", np.round(best_theta, 2))
for sub, verb, y in data:
    p = model(best_theta, sub, verb)
    print(f"  label={y:+d}, pred={p:+.2f}, correct={'yes' if np.sign(p)==y else 'no'}")

### Activity 11.3

Add a new test sentence that the classifier has never seen and predict its class. Try (1.05, 0.25) as subject and (2.05, 1.05) as verb. Is the sign positive as expected?

In [ ]:
# TODO: predict on an unseen sentence.
test_sub  = None
test_verb = None

if test_sub is not None and test_verb is not None:
    pred = model(best_theta, test_sub, test_verb)
    print(f"pred = {pred:+.3f}  ->  class {'positive' if pred > 0 else 'negative'}")
else:
    print("Fill in test_sub and test_verb, then re-run.")

<details><summary>Solution</summary>

```python
test_sub  = (1.05, 0.25)
test_verb = (2.05, 1.05)
```
Prediction should be positive (~+0.7) because it lies near the training positives.
</details>

---
## Section summary

* Words become quantum states through angle encoding, and semantic similarity is the quantum inner product.
* Sentence structure can be encoded through entangling gates that mirror grammatical roles (DisCoCat intuition).
* Attention has a natural quantum implementation via the swap test, which computes overlaps in one shot.
* Chaining these pieces gives a tiny quantum sentence classifier that trains like a variational quantum circuit.

---
## End-of-Unit Quiz (10 multiple choice)

**Q1.** Angle encoding of a 2D vector (a, b) produces the qubit state:

A. cos(a) |0> + sin(b) |1>
B. cos(a/2) |0> + e^{i b} sin(a/2) |1>
C. a |0> + b |1>
D. (a + b) |+>

**Q2.** In quantum NLP, semantic similarity between two word states |w1> and |w2> is measured by:

A. Their Hamming distance
B. |<w1|w2>|^2, the squared inner product
C. The sum of their entries
D. The number of qubits used

**Q3.** DisCoCat represents a sentence by:

A. A single large gate matrix
B. A tensor contraction that follows the sentence's grammar
C. A softmax over a bag of words
D. A binary string

**Q4.** The swap test outputs:

A. Whether two states are orthogonal, with certainty
B. A probability that encodes the overlap |<a|b>|^2
C. A copy of the second state
D. The sum of two states

**Q5.** Why is attention "quantum-friendly"?

A. It only requires classical addition
B. Its core is an inner product, which quantum states compute natively
C. It has no matrix multiplication
D. It uses only integer weights

**Q6.** In our tiny sentence classifier, the model output for classification was:

A. The probability of measuring |00>
B. <Z> on the first qubit
C. The largest amplitude
D. The Hamming weight

**Q7.** What role did the CNOT play in the sentence_state construction?

A. It measured the qubits
B. It entangled the subject and verb wires, encoding a grammatical link
C. It reset the state
D. It computed the loss

**Q8.** Two nearby vectors in the (a, b) angle encoding produce states that are:

A. Orthogonal
B. Highly overlapping
C. Mixed states
D. Undefined

**Q9.** A key advantage of quantum feature maps for language is:

A. They eliminate the need for training data
B. They can encode exponentially large feature spaces in few qubits
C. They avoid all noise
D. They remove the need for classical optimization

**Q10.** Which classical AI concept most directly inspires the DisCoCat approach?

A. Convolutional filters
B. Compositional semantics via tensor products
C. Reinforcement learning
D. Batch normalization

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 11!")